# Supply Chain Data Model - Inventory & Purchase Orders

## Schema Structure
- **Inventory Table**: Current stock levels and warehouse locations
- **InventoryTransactions Table**: All stock movements (receipts, sales, adjustments)
- **PurchaseOrders Table**: Order headers with supplier and timing information
- **PurchaseOrderItems Table**: Line items with product details and quantities
- Supports real-time inventory tracking with purchase order integration

In [ ]:
################################################################################################
# Schema Configuration - You can define different value here
################################################################################################

# Schema Configuration
SCHEMA_NAME = "supplychain"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA_NAME}")
print(f"✅ {SCHEMA_NAME} schema ready!")

In [ ]:
# Create all Inventory and Purchase Order tables - Inventory, InventoryTransactions, PurchaseOrders, PurchaseOrderItems

# 1. Create Inventory current levels table
TABLE_NAME = "Inventory"
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA_NAME}.{TABLE_NAME} (
    InventoryID INT,
    ProductID INT,             -- Links to existing Product table
    ProductName STRING,        -- Denormalized for usability
    ProductCategory STRING,    -- Camping, Kitchen, Ski
    WarehouseLocation STRING,  -- Main, Backup, Regional
    CurrentStock INT,          -- Units currently available
    ReservedStock INT,         -- Units allocated to orders
    AvailableStock INT,        -- CurrentStock - ReservedStock
    SafetyStockLevel INT,      -- Minimum stock threshold
    ReorderPoint INT,          -- When to trigger new purchase order
    MaxStockLevel INT,         -- Maximum warehouse capacity
    LastUpdated TIMESTAMP,     -- When stock level last changed
    AverageCost DECIMAL(10,2), -- Weighted average unit cost
    Status STRING,             -- Active, LowStock, OutOfStock, Excess
    CreatedBy STRING,          -- System user
    CreatedDate TIMESTAMP      -- Record creation
)
USING DELTA
"""
spark.sql(create_table_sql)
print(f"✅ {SCHEMA_NAME}.{TABLE_NAME} table created!")

# 2. Create InventoryTransactions audit trail table
TABLE_NAME = "InventoryTransactions"
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA_NAME}.{TABLE_NAME} (
    TransactionID INT,
    ProductID INT,             -- Links to existing Product table
    ProductName STRING,        -- Denormalized for usability
    ProductCategory STRING,    -- Camping, Kitchen, Ski
    WarehouseLocation STRING,  -- Where transaction occurred
    TransactionType STRING,    -- Receipt, Sale, Transfer, Adjustment, Damage, Return
    TransactionDate TIMESTAMP, -- When transaction occurred
    Quantity INT,              -- Units moved (positive = in, negative = out)
    UnitCost DECIMAL(10,2),    -- Cost per unit for this transaction
    TotalValue DECIMAL(12,2),  -- Quantity * UnitCost
    ReferenceNumber STRING,    -- PO number, sale ID, etc.
    ReasonCode STRING,         -- Why transaction occurred
    StockBefore INT,           -- Inventory level before transaction
    StockAfter INT,            -- Inventory level after transaction
    ProcessedBy STRING,        -- System user who processed
    Notes STRING,              -- Additional details
    CreatedBy STRING,          -- System user
    CreatedDate TIMESTAMP      -- Record creation
)
USING DELTA
"""
spark.sql(create_table_sql)
print(f"✅ {SCHEMA_NAME}.{TABLE_NAME} table created!")

# 3. Create PurchaseOrders header table
TABLE_NAME = "PurchaseOrders"
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA_NAME}.{TABLE_NAME} (
    PurchaseOrderID INT,
    PurchaseOrderNumber STRING,  -- PO-YYYY-NNNN format
    SupplierID INT,             -- Links to Suppliers table
    SupplierName STRING,        -- Denormalized for usability
    OrderDate DATE,             -- When order was placed
    ExpectedDeliveryDate DATE,  -- Supplier commitment date
    ActualDeliveryDate DATE,    -- NULL until delivered
    Status STRING,              -- Draft, Sent, Confirmed, InTransit, Delivered, Cancelled
    TotalOrderValue DECIMAL(12,2), -- Sum of all line items
    DeliveryLocation STRING,    -- Warehouse/distribution center
    OrderedBy STRING,           -- System user who placed order
    Priority STRING,            -- Low, Medium, High, Emergency
    Notes STRING,               -- Special instructions or comments
    CreatedBy STRING,           -- System user
    CreatedDate TIMESTAMP       -- Record creation
)
USING DELTA
"""
spark.sql(create_table_sql)
print(f"✅ {SCHEMA_NAME}.{TABLE_NAME} table created!")

# 4. Create PurchaseOrderItems line item table
TABLE_NAME = "PurchaseOrderItems"
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA_NAME}.{TABLE_NAME} (
    PurchaseOrderItemID INT,
    PurchaseOrderID INT,        -- Links to PurchaseOrders table
    PurchaseOrderNumber STRING, -- Denormalized for usability
    ProductID INT,              -- Links to existing Product table
    ProductName STRING,         -- Denormalized for usability
    ProductCategory STRING,     -- Camping, Kitchen, Ski
    QuantityOrdered INT,        -- Units requested
    QuantityReceived INT,       -- Units actually delivered (NULL until received)
    UnitCost DECIMAL(10,2),     -- Cost per unit from supplier
    LineTotal DECIMAL(12,2),    -- QuantityOrdered * UnitCost
    Status STRING,              -- Pending, Backordered, Shipped, Received, Cancelled
    ExpectedDate DATE,          -- When this line item expected
    ReceivedDate DATE,          -- When this line item actually received
    Notes STRING,               -- Product-specific notes
    CreatedBy STRING,           -- System user
    CreatedDate TIMESTAMP       -- Record creation
)
USING DELTA
"""
spark.sql(create_table_sql)
print(f"✅ {SCHEMA_NAME}.{TABLE_NAME} table created!")

print("\n🎉 All Inventory and Purchase Order tables created successfully!")